In [ ]:
# RabindricGPT — One-Cell Idempotent Colab Bootstrap & Supervisor
# Automates Drive mounting, git synchronization, artifact extraction, preflight gating, and L4 training execution.

import os
import sys
import shutil
from pathlib import Path

# 1. Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print(f"Drive mount skipped or already mounted: {e}")

# 2. Idempotent git clone or fast-forward pull
repo_path = Path('/content/RabindricGPT')
if not repo_path.exists():
    !git clone https://github.com/ASMITSARKAR/RabindricGPT.git /content/RabindricGPT
else:
    !git -C /content/RabindricGPT pull --ff-only

os.chdir('/content/RabindricGPT')
if '/content/RabindricGPT' not in sys.path:
    sys.path.insert(0, '/content/RabindricGPT')

# 3. Setup persistent Drive folder layout and symlink runs
drive_base = Path('/content/drive/MyDrive/RabindricGPT')
drive_runs = drive_base / 'runs'
drive_prep = drive_base / 'prepared'
drive_runs.mkdir(parents=True, exist_ok=True)
drive_prep.mkdir(parents=True, exist_ok=True)

local_runs = Path('/content/RabindricGPT/runs')
if local_runs.is_symlink() or local_runs.exists():
    if local_runs.is_symlink():
        local_runs.unlink()
    else:
        shutil.rmtree(local_runs)
os.symlink(str(drive_runs), str(local_runs))

# 4. Extract shards and tokenizers from Drive prepared/ folder if not yet unpacked
for archive in ['tok.zip', 'shards.zip']:
    # Check prepared/ first, then drive root
    src = drive_prep / archive if (drive_prep / archive).exists() else (drive_base / archive)
    folder = archive.replace('.zip', '')
    if src.exists() and not (Path('/content/RabindricGPT') / folder).exists():
        print(f"Unpacking {src} into /content/RabindricGPT/...")
        !unzip -q -o "{src}" -d /content/RabindricGPT/

# 5. Install dependencies
!pip install -q sentencepiece pyyaml

# 6. Run quick CPU preflight gating
!python scripts/00_preflight.py --quick

# 7. Launch supervised queue runner on L4 GPU
!python colab/runner_lib.py --queue configs/runs/queue.yaml